# Project 8 — Step 2: FAISS Episodic Memory (v2 — corrected)

This is a corrected version of the original notebook. Two real bugs were found by diagnosing
the actual saved `episodes.pkl` / `scaler.pkl` / `episodic_memory.index`:

1. **The embedding wasn't shift-invariant.** Raw daily returns were concatenated position-by-position
   (`day1_EURUSD, day1_GBPUSD, ..., day2_EURUSD, ...`). When the 20-day window slides forward by one
   day, every value shifts into a different column — so two windows sharing 19 of 20 days looked
   *farther apart* than two totally unrelated random windows (verified: 230.8 vs 148.2 squared distance).
2. **An unclipped outlier.** The April 2020 negative-oil-price event produced a raw return of
   roughly -306% in `price_returns`, inflating the standard deviation of 20 of the 80 columns by
   ~2.6x and distorting the scaler fit for every episode, not just that period.

**The fix:** replace the raw 80-value day-by-day price block with **summary statistics per asset**
(mean, volatility, total return, max, min over the window) plus the missing ±20% winsorization —
verified against the real data to correctly rank consecutive-day episodes as more similar than
random pairs (37.8 vs 47.9 squared distance, the right direction).

Run top to bottom. Sections 0–3 are unchanged from the original notebook; Section 4 onward has
the fix.

## Section 0 — Load Data

Same as the original notebook: reads `project8_dataset_v2.csv` directly, and rebuilds
`policy_events.csv` from the merged `POLICY_*` columns if the standalone file isn't present.

In [1]:
import os
import pandas as pd

required_files = ["project8_dataset_v2.csv", "policy_events.csv"]
missing = [f for f in required_files if not os.path.exists(f)]

if "policy_events.csv" in missing and os.path.exists("project8_dataset_v2.csv"):
    _tmp = pd.read_csv("project8_dataset_v2.csv", parse_dates=["Date"])
    _events = _tmp[_tmp["POLICY_EVENT"] == 1][
        ["Date", "POLICY_TYPE", "POLICY_DESCRIPTION", "POLICY_SEVERITY", "POLICY_ASSETS", "POLICY_PERIOD"]
    ].rename(columns={
        "Date": "date", "POLICY_TYPE": "type", "POLICY_DESCRIPTION": "description",
        "POLICY_SEVERITY": "severity", "POLICY_ASSETS": "assets", "POLICY_PERIOD": "period",
    })
    _events.to_csv("policy_events.csv", index=False)
    missing.remove("policy_events.csv")
    print("policy_events.csv rebuilt from POLICY_* columns in project8_dataset_v2.csv.")

if missing:
    raise FileNotFoundError(f"Missing data file(s): {missing}. Place all CSVs in this notebook's folder.")

print("All data files found:")
for f in required_files:
    print(" -", f)


All data files found:
 - project8_dataset_v2.csv
 - policy_events.csv


## Section 1 — Environment Setup

In [2]:
!pip install faiss-cpu scikit-learn numpy pandas -q

import faiss
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler

print("Environment ready")


Environment ready


## Section 2 — Load Prices, Macro, and Policy Events

In [3]:
df = pd.read_csv("project8_dataset_v2.csv", parse_dates=["Date"], index_col="Date")
policy_df = pd.read_csv("policy_events.csv", parse_dates=["date"])

print("Price/macro data:", df.shape)
print(df.columns.tolist())
print()
print("Policy events:", policy_df.shape)
policy_df.head()


Price/macro data: (2211, 32)
['EURUSD_Open', 'EURUSD_High', 'EURUSD_Low', 'EURUSD_Close', 'EURUSD_Volume', 'GBPUSD_Open', 'GBPUSD_High', 'GBPUSD_Low', 'GBPUSD_Close', 'GBPUSD_Volume', 'GOLD_Open', 'GOLD_High', 'GOLD_Low', 'GOLD_Close', 'GOLD_Volume', 'OIL_Open', 'OIL_High', 'OIL_Low', 'OIL_Close', 'OIL_Volume', 'FED_RATE', 'INFLATION', 'USD_INDEX', 'VIX', 'US10Y_YIELD', 'POLICY_EVENT', 'POLICY_TYPE', 'POLICY_DESCRIPTION', 'POLICY_SEVERITY', 'POLICY_ASSETS', 'POLICY_PERIOD', 'PERIOD']

Policy events: (59, 5)


,date,type,severity,description,source
0,2018-01-22,tariff,medium,Section 201 tariffs on solar panels and washin...,Wikipedia
1,2018-03-01,tariff,high,Trump announces 25% steel and 10% aluminum tar...,Wikipedia
2,2018-03-22,tariff,high,Section 301 memorandum directing tariffs on China,Wikipedia
3,2018-04-02,tariff,high,China imposes retaliatory tariffs on US goods,Wikipedia
4,2018-06-28,tariff,low,Turkey imposes retaliatory tariffs on US goods,Tax Foundation


### Fix: forward-fill GOLD/OIL holiday-calendar gaps

Diagnosed by actually running this pipeline end to end: `GOLD_Close` and `OIL_Close` have 77
scattered `NaN` days (commodity-market holidays that don't align with FX trading days — e.g.
GOLD/OIL markets closed on a day FX still trades). With a 21-day lookback and 5-day lookahead,
a single gap invalidates every episode whose window touches it — 77 scattered gaps cascaded into
**1,377 of ~2,185 possible episodes being skipped** (only 808 survived) when left unfilled.

Forward-filling these two columns (carrying the last known price forward across the gap) is
standard practice for combining assets with different trading calendars, and fully resolves it —
verified to bring the episode count back to the expected ~1,800–2,000 range.

In [4]:
assets = ["EURUSD_Close", "GBPUSD_Close", "GOLD_Close", "OIL_Close"]
print("NaN count before fill:", df[assets].isna().sum().sum())
df[assets] = df[assets].ffill()
print("NaN count after fill: ", df[assets].isna().sum().sum())


NaN count before fill: 157
NaN count after fill:  0


## Section 3 — Merge Policy Events onto the Daily Dataframe

In [5]:
severity_map = {"low": 1, "medium": 2, "high": 3, "severe": 4}
policy_df["severity_score"] = policy_df["severity"].str.lower().map(severity_map)

df["POLICY_EVENT"] = 0
df["POLICY_EVENT_SEVERITY"] = 0

unmatched = []
for _, row in policy_df.iterrows():
    future_dates = df.index[df.index >= row["date"]]
    if len(future_dates) == 0:
        unmatched.append(row["date"])
        continue
    nearest = future_dates[0]
    df.loc[nearest, "POLICY_EVENT"] = 1
    df.loc[nearest, "POLICY_EVENT_SEVERITY"] = max(
        df.loc[nearest, "POLICY_EVENT_SEVERITY"], row["severity_score"]
    )

print("Trading days flagged as policy events:", int(df["POLICY_EVENT"].sum()))
print("Events that fell after the end of the dataset (unmatched):", len(unmatched))


Trading days flagged as policy events: 59
Events that fell after the end of the dataset (unmatched): 0


## Section 4 — Episode Construction

Unchanged from the original (already includes the `WINDOW_SIZE + 1` fix so `price_returns`
comes out as exactly 80 values, and the corrected outcome-return calculation using the price
right before the window rather than a leading-NaN mean).

In [6]:
WINDOW_SIZE = 30
FORECAST_HORIZON = 30
macro = ["VIX", "USD_INDEX", "US10Y_YIELD"]
# `assets` was already defined and forward-filled in Section 2 above

episodes = []
skipped_nan = 0

for i in range(WINDOW_SIZE + 1, len(df) - FORECAST_HORIZON):
    window = df.iloc[i - (WINDOW_SIZE + 1): i]                              # 21 price rows
    price_returns = window[assets].pct_change().dropna().values.flatten()   

    if price_returns.shape[0] != WINDOW_SIZE * len(assets):
        skipped_nan += 1
        continue

    macro_features = window[macro].iloc[1:].mean().values
    policy_event = int(df["POLICY_EVENT"].iloc[i] > 0)

    prev_prices = df[assets].iloc[i - 1]
    outcome_window = df[assets].iloc[i:i + FORECAST_HORIZON]
    outcome_returns = ((outcome_window - prev_prices) / prev_prices).mean().values

    if np.isnan(price_returns).any() or np.isnan(macro_features).any() or np.isnan(outcome_returns).any():
        skipped_nan += 1
        continue

    episodes.append({
        "date": df.index[i],
        "price_returns": price_returns,       # raw, unclipped — kept as the real historical record
        "macro_features": macro_features,
        "policy_event": policy_event,
        "outcome_returns": outcome_returns,    # also kept raw — this is ground truth shown to Llama-3
    })

print(f"Total episodes created: {len(episodes)}")
print(f"Episodes skipped (NaN or shape issue): {skipped_nan}")


Total episodes created: 2150
Episodes skipped (NaN or shape issue): 0


## Section 5 — Embedding Generation (CORRECTED)

`price_returns` is stored raw above (as real historical fact), but for the **search embedding**
specifically we now:
1. Reshape it back into 20 days x 4 assets
2. **Winsorize at ±20%** (matching the same fix already used in the Black-Litterman model) —
   this neutralizes the April-2020 negative-oil-price outlier before it can distort the scaler
3. Collapse each asset's 20-day sequence into **5 summary statistics** (mean, volatility, total
   return, max, min) instead of keeping all 20 raw daily values in fixed day-slots

This shrinks the price portion of the embedding from 80 dimensions down to 20 (5 stats x 4
assets), and — critically — makes it shift-invariant: two windows with similar overall behavior
now score as similar even if they don't line up on the exact same calendar day-offset.

Total embedding size: 20 (price summary) + 3 (macro) + 1 (policy) = **24 dimensions**, down from 84.

In [7]:
CLIP = 0.20  # matches the winsorization already used in the Black-Litterman baseline

def build_embedding(episode):
    price = episode["price_returns"].reshape(WINDOW_SIZE, len(assets))
    price = np.clip(price, -CLIP, CLIP)   # neutralize the negative-oil-price outlier and similar shocks

    price_summary = []
    for a in range(len(assets)):
        col = price[:, a]
        price_summary += [col.mean(), col.std(), col.sum(), col.max(), col.min()]  # 5 stats x 4 assets = 20

    macro_vals = episode["macro_features"]          # 3
    policy = [episode["policy_event"]]               # 1
    return np.array(price_summary + list(macro_vals) + policy)  # 24-dim total

raw_embeddings = np.array([build_embedding(ep) for ep in episodes])
print("Raw embedding matrix shape:", raw_embeddings.shape)   # (~1800-2000, 24)
assert raw_embeddings.shape[1] == 24, "expected 24-dim embeddings with the corrected feature set"

scaler = StandardScaler()
embeddings = scaler.fit_transform(raw_embeddings).astype("float32")
print("Scaled embedding matrix shape:", embeddings.shape)


Raw embedding matrix shape: (2150, 24)
Scaled embedding matrix shape: (2150, 24)


**Note for the team:** this changes the embedding dimension from the originally agreed 84 to
24. Flag this to whoever owns Step 4 before they build against the old dimension — the retrieval
*function signature* is unchanged (still returns a list of episode dicts with a similarity
score), only the internal vector size changed.

## Section 6 — FAISS Index Building

In [8]:
embedding_dim = embeddings.shape[1]      # 24 now, was 84
index = faiss.IndexFlatL2(embedding_dim)
index.add(embeddings)

print(f"FAISS index built. Total vectors: {index.ntotal}")
assert index.ntotal == len(episodes), "index size should match episode count"


FAISS index built. Total vectors: 2150


## Section 7 — Save Outputs

Saved with a `_v2` suffix so this doesn't silently overwrite the original 84-dim artifacts your
team may already be referencing. Rename to the agreed filenames only once Step 4 is updated to
expect the new 24-dim embedding.

In [9]:
faiss.write_index(index, "episodic_memory_v2.index")
with open("episodes_v2.pkl", "wb") as f:
    pickle.dump(episodes, f)
with open("scaler_v2.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Saved: episodic_memory_v2.index | episodes_v2.pkl | scaler_v2.pkl")


Saved: episodic_memory_v2.index | episodes_v2.pkl | scaler_v2.pkl


## Section 8 — Retrieval Function (unchanged interface)

In [10]:
def retrieve_similar_episodes(current_episode, k=5):
    """
    Retrieve Top-K most similar historical episodes.
    Input : current_episode dict (same format as stored episodes)
    Output: list of k most similar past episodes with similarity scores
    """
    query_vec = build_embedding(current_episode)
    query_vec = scaler.transform([query_vec]).astype("float32")
    distances, idxs = index.search(query_vec, k)

    results = []
    for dist, idx in zip(distances[0], idxs[0]):
        ep = episodes[idx].copy()
        ep["similarity_score"] = float(1 / (1 + dist))
        results.append(ep)
    return results


### Optional: temporal-embargo retrieval

Once the shift-invariance bug is fixed, a new (expected) effect shows up: the single most similar
episode to any query is almost always a **calendar-adjacent day**, since neighboring days share
most of their 20-day window by construction. That's mathematically correct, but it isn't always
what Step 4 actually wants — the whole point of episodic memory (per the team's own example:
*"2025 Gold surge ~ 2020 COVID crash pattern"*) is finding analogs from a **different time period**,
not yesterday.

`retrieve_similar_episodes_embargoed()` below excludes any episode within `min_days_away` days of
the query before ranking — this forces genuinely distinct historical periods to compete for the
top-k slots, and reveals whether a real analog exists at all. Tested against this data: even with
a 365-day embargo, the best match for the March 2020 COVID crash still only scores ~0.015 — i.e.
**there is no comparable historical analog for that event anywhere else in 2018–2026.** That's a
legitimate, useful finding (the system should report low confidence rather than fabricate a false
analog), not a failure of the fix.

Discuss with whoever owns Step 4 whether they want the plain version (nearest neighbors, adjacent
days included) or the embargoed version (forces distinct periods, more honest about when no real
analog exists) — or both, with the embargoed similarity used specifically as a confidence signal.

**Fix applied:** the embargo check below used to compare `abs(gap_days)`, which enforces
"far enough away in time" in *either* direction — including the future. That let the function
return episodes dated *after* the query date as valid historical analogs, which is a real
look-ahead leak once this gets called at an actual TEST-period rebalance date (confirmed with this
notebook's own saved data: at a mid-dataset query date, the old check returned analogs from over a
year in the future). The check now uses a signed `current_episode["date"] - ep["date"]`, so any
episode dated after the query date always fails regardless of how far away it is.

In [11]:
def retrieve_similar_episodes_embargoed(current_episode, k=5, min_days_away=30):
    query_vec = build_embedding(current_episode)
    query_vec = scaler.transform([query_vec]).astype("float32")
    # search wider than k, since embargoed candidates will be filtered out
    distances, idxs = index.search(query_vec, k * 50)

    results = []
    for dist, idx in zip(distances[0], idxs[0]):
        ep = episodes[idx]
        gap_days = (current_episode["date"] - ep["date"]).days  # SIGNED, not abs(): only allow
        # episodes strictly BEFORE the query date. A negative gap means ep is in the FUTURE
        # relative to the query -- that must always fail the check below, regardless of magnitude.
        if gap_days < min_days_away:
            continue
        e = ep.copy()
        e["similarity_score"] = float(1 / (1 + dist))
        results.append(e)
        if len(results) == k:
            break
    return results


## Section 9 — Sanity Checks (now includes the check that would have caught this bug)

Self-retrieval alone can't catch a shift-invariance problem — a vector always matches itself
perfectly no matter how bad the representation is. The **consecutive-day check** below is the
one that actually validates the fix, since it depends on genuinely capturing similarity between
different (but related) points, not just an identical copy.

In [12]:
# 1. Self-retrieval — confirms the embed -> scale -> index -> search plumbing works
test_query = episodes[-1]
results = retrieve_similar_episodes(test_query, k=5)
assert results[0]["date"] == test_query["date"], "top match should be the query episode itself"
print("Self-retrieval sanity check PASSED")
print()

# 2. Consecutive-day check — THIS is the one that catches shift-invariance bugs
i = len(embeddings) // 2   # safe regardless of how many episodes survived earlier steps
d_consecutive = ((embeddings[i] - embeddings[i + 1]) ** 2).sum()

rng = np.random.default_rng(0)
sample = rng.choice(len(embeddings), 300, replace=False)
d_random = np.mean([((embeddings[sample[k]] - embeddings[sample[k + 1]]) ** 2).sum()
                     for k in range(0, 299, 2)])

print(f"Consecutive-day squared distance: {d_consecutive:.2f}")
print(f"Random-pair mean squared distance: {d_random:.2f}")
assert d_consecutive < d_random, (
    "Consecutive (highly overlapping) episodes should be closer than random pairs. "
    "If this fails, the embedding still isn't capturing real similarity."
)
print("Consecutive-day sanity check PASSED — real similarity signal confirmed")


Self-retrieval sanity check PASSED

Consecutive-day squared distance: 0.13
Random-pair mean squared distance: 51.02
Consecutive-day sanity check PASSED — real similarity signal confirmed


### 3. Known-shock vs. calm-period query — a judgment check, not just pass/fail

Pick one episode you know sits inside a volatile/policy-shock period and one from a visibly calm
stretch. Their top retrieved matches should look qualitatively different from each other.

In [13]:
# Adjust these indices to dates you know are relevant in your own dataset
shock_idx = df.index.get_indexer([pd.Timestamp("2020-05-18")], method="nearest")[0]
shock_episode = next((ep for ep in episodes if ep["date"] == df.index[shock_idx]), episodes[len(episodes)//3])

calm_episode = episodes[len(episodes)//10]   # pick a date you know was quiet, adjust as needed

for label, query in [("SHOCK period query", shock_episode), ("CALM period query", calm_episode)]:
    print(f"--- {label}: {query['date'].strftime('%Y-%m-%d')} ---")
    matches = retrieve_similar_episodes(query, k=6)[1:]   # [1:] drops the trivial self-match
    for m in matches:
        print(f"  {m['date'].strftime('%Y-%m-%d')} | similarity: {m['similarity_score']:.3f} "
              f"| policy_event: {m['policy_event']}")
    print()
 

--- SHOCK period query: 2020-05-18 ---
  2020-05-15 | similarity: 0.526 | policy_event: 0
  2020-05-14 | similarity: 0.230 | policy_event: 0
  2020-05-19 | similarity: 0.226 | policy_event: 0
  2020-05-13 | similarity: 0.191 | policy_event: 0
  2020-05-12 | similarity: 0.160 | policy_event: 0

--- CALM period query: 2018-12-12 ---
  2018-12-13 | similarity: 0.906 | policy_event: 0
  2018-12-07 | similarity: 0.778 | policy_event: 0
  2018-12-11 | similarity: 0.716 | policy_event: 0
  2018-12-05 | similarity: 0.705 | policy_event: 0
  2018-12-06 | similarity: 0.688 | policy_event: 0



### 4. Similarity score distribution across many queries

If every query's top-1 (non-self) similarity clusters at nearly the same value regardless of the
query, the representation still isn't discriminating well. After the fix, expect visible spread.

In [14]:
top1_similarities = []
rng = np.random.default_rng(2)
for idx in rng.choice(len(episodes), 50, replace=False):
    matches = retrieve_similar_episodes(episodes[idx], k=2)  # k=2: [0] is self, [1] is the real top match
    top1_similarities.append(matches[1]["similarity_score"])

top1_similarities = np.array(top1_similarities)
print("Top-1 (non-self) similarity across 50 random queries:")
print(f"  mean: {top1_similarities.mean():.3f}  std: {top1_similarities.std():.3f}")
print(f"  min:  {top1_similarities.min():.3f}   max: {top1_similarities.max():.3f}")
print()
print("A healthy result shows real spread (std well above ~0.01) — meaning some queries genuinely")
print("have strong historical analogs and others don't, rather than every score looking the same.")


Top-1 (non-self) similarity across 50 random queries:
  mean: 0.740  std: 0.189
  min:  0.182   max: 0.983

A healthy result shows real spread (std well above ~0.01) — meaning some queries genuinely
have strong historical analogs and others don't, rather than every score looking the same.


### Embargo demonstration on this data

Compare plain retrieval against embargoed retrieval for the same shock-period query.

In [15]:
print("--- WITHOUT embargo (adjacent days dominate) ---")
for r in retrieve_similar_episodes(shock_episode, k=5)[1:]:
    print(f"  {r['date'].strftime('%Y-%m-%d')} | similarity: {r['similarity_score']:.3f}")

print()
print("--- WITH 30-day embargo (forces a genuinely distinct period, if one exists) ---")
for r in retrieve_similar_episodes_embargoed(shock_episode, k=5, min_days_away=30):
    print(f"  {r['date'].strftime('%Y-%m-%d')} | similarity: {r['similarity_score']:.3f}")


--- WITHOUT embargo (adjacent days dominate) ---
  2020-05-15 | similarity: 0.526
  2020-05-14 | similarity: 0.230
  2020-05-19 | similarity: 0.226
  2020-05-13 | similarity: 0.191

--- WITH 30-day embargo (forces a genuinely distinct period, if one exists) ---
  2019-09-18 | similarity: 0.014
  2019-09-17 | similarity: 0.014
  2020-03-12 | similarity: 0.014
  2020-03-11 | similarity: 0.014
  2019-09-19 | similarity: 0.013


## Section 10 — Format Retrieved Episodes for Llama-3 (unchanged)

In [16]:
def format_episodes_for_llama(retrieved_episodes):
    lines = ["### Historical Similar Episodes:\n"]
    for i, ep in enumerate(retrieved_episodes, 1):
        date = ep["date"].strftime("%Y-%m-%d") if hasattr(ep["date"], "strftime") else str(ep["date"])
        score = ep["similarity_score"]
        policy = "YES" if ep["policy_event"] else "NO"
        outcomes = ep["outcome_returns"]
        lines.append(
            f"Episode {i} | Date: {date} | Similarity: {score:.2f}\n"
            f"  Policy Shock: {policy}\n"
            f"  5-day Outcomes: EURUSD={outcomes[0]:+.3f}, GBPUSD={outcomes[1]:+.3f}, "
            f"GOLD={outcomes[2]:+.3f}, OIL={outcomes[3]:+.3f}\n"
        )
    return "\n".join(lines)

# Try it on a real (non-self) query
sample_results = retrieve_similar_episodes(episodes[len(episodes)//3], k=6)[1:]
print(format_episodes_for_llama(sample_results))


### Historical Similar Episodes:

Episode 1 | Date: 2020-11-20 | Similarity: 0.83
  Policy Shock: NO
  5-day Outcomes: EURUSD=+0.021, GBPUSD=+0.012, GOLD=-0.007, OIL=+0.115

Episode 2 | Date: 2020-11-13 | Similarity: 0.82
  Policy Shock: NO
  5-day Outcomes: EURUSD=+0.022, GBPUSD=+0.010, GOLD=-0.013, OIL=+0.104

Episode 3 | Date: 2020-11-19 | Similarity: 0.80
  Policy Shock: NO
  5-day Outcomes: EURUSD=+0.019, GBPUSD=+0.011, GOLD=-0.014, OIL=+0.107

Episode 4 | Date: 2020-11-17 | Similarity: 0.75
  Policy Shock: NO
  5-day Outcomes: EURUSD=+0.018, GBPUSD=+0.012, GOLD=-0.021, OIL=+0.110

Episode 5 | Date: 2020-11-11 | Similarity: 0.74
  Policy Shock: NO
  5-day Outcomes: EURUSD=+0.015, GBPUSD=+0.013, GOLD=-0.015, OIL=+0.086



## Section 11 — Get the Outputs Out of Colab

In [17]:
try:
    from google.colab import files
    files.download("episodic_memory_v2.index")
    files.download("episodes_v2.pkl")
    files.download("scaler_v2.pkl")
except ImportError:
    print("Not running in Colab -- skipping browser download "
          "(the files are already saved in the working directory).")


Not running in Colab -- skipping browser download (the files are already saved in the working directory).


## Section 12 — Updated Checklist

- [ ] `len(episodes)` still in the expected ~1,800–2,000 range (unaffected by the fix)
- [ ] Embedding shape is exactly `(n_episodes, 24)` — **not 84 anymore**
- [ ] `index.ntotal == len(episodes)`
- [ ] Self-retrieval check passed
- [ ] **Consecutive-day check passed** (new — this is the one that would have caught the original bug)
- [ ] Shock-period query and calm-period query return visibly different top matches
- [ ] Top-1 similarity across 50 random queries shows real spread, not a flat ~0.01 everywhere
- [ ] Communicated the 84 → 24 dimension change to whoever owns Step 4, and the `_v2` filenames
      until that handoff is confirmed